**BHASHINI USER DATA ANALYSIS**

Data loading

In [3]:
import pandas as pd

df1 = pd.read_excel(r"C:\Users\sneha\Downloads\BHASHHINI\BhashaDaan_Data.xlsx", sheet_name="User Registerations")

df2 = pd.read_excel(r"C:\Users\sneha\Downloads\BHASHHINI\BhashaDaan_Data.xlsx", sheet_name="Contribution Counts")
df2.head() 
df1.head() 

,Activity_date,New_users,Contributors
0,2026-01-17,13,7
1,2026-01-18,15,8
2,2026-01-19,17,11
3,2026-01-20,19,8
4,2026-01-21,25,15


In [44]:
import pandas as pd
import plotly.express as px

# Ensure datetime and month period
df1['Activity_date'] = pd.to_datetime(df1['Activity_date'])
df1['Month_Year'] = df1['Activity_date'].dt.to_period('M')

# Aggregate monthly totals
monthly_totals = df1.groupby('Month_Year')[['New_users','Contributors']].sum().reset_index()

# Conversion rate (%)
monthly_totals['Conversion_rate'] = (monthly_totals['Contributors'] / monthly_totals['New_users']) * 100

# ✅ Sort by month properly
monthly_totals = monthly_totals.sort_values('Month_Year')

# Convert Month_Year to string for plotting
monthly_totals['Month_Year'] = monthly_totals['Month_Year'].astype(str)

# Plot line chart with percentage labels
fig = px.line(
    monthly_totals,
    x="Month_Year",
    y="Conversion_rate",
    text="Conversion_rate",
    markers=True,
    title="Monthly Contributor-to-User Conversion Rate"
)

# Show percentages clearly
fig.update_traces(texttemplate='%{text:.1f}%', textposition='top center')

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Conversion Rate (%)",
    yaxis_tickformat=".1f",
    height=500
)

fig.show()




In [7]:
# Group contributions by date and language
daily_lang_contrib = df2.groupby(['Activity_date', 'Language'])['Contributions'].sum().reset_index()
# Pivot so each language is a column
pivot_daily = daily_lang_contrib.pivot(index='Activity_date', columns='Language', values='Contributions').fillna(0)
# Merge registrations with contributions on Activity_date
merged = pd.merge(df1[['Activity_date','Contributors']], 
                  daily_lang_contrib, 
                  on='Activity_date', 
                  how='left')


In [14]:
# Convert to datetime if not already
df2['Activity_date'] = pd.to_datetime(df2['Activity_date'])
df2['Month_Year'] = df2['Activity_date'].dt.strftime('%B %Y')
monthly_lang_contrib = df2.groupby(['Month_Year','Language'])['Contributions'].sum().reset_index()


In [15]:
import plotly.express as px
import plotly.graph_objects as go

fig = px.bar(
    daily_lang_contrib,
    x='Activity_date',
    y='Contributions',
    color='Language',
    title="Daily Contributions by Language",
    barmode='stack'   # or 'group' for side-by-side
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Contributions",
    height=600
)

# Show in Jupyter / plain Python
fig.show()


In [78]:
import plotly.express as px

monthly_lang_contrib = df2.groupby(['Month_Year','Language'])['Contributions'].sum().reset_index()

fig_month = px.bar(
    monthly_lang_contrib,
    x='Contributions',
    y='Month_Year',
    color='Language',
    text='Language',
    height=900,
    width=1000,   # horizontal bars
    barmode='stack',
    title="Monthly Contributions by Language"
)

# ✅ Auto text placement
fig_month.update_traces(textposition='auto')

# ✅ Set x-axis range 0–5000 with divisions of 500
fig_month.update_layout(
    xaxis=dict(
        range=[0, 5000],   # axis goes from 0 to 5000
        dtick=500,         # tick marks every 500
        showgrid=True,     # optional: show gridlines at each tick
        gridcolor="lightgrey"
    )
)

fig_month.show()


In [82]:
import pandas as pd
import plotly.express as px

# Convert Month_Year strings like "January 2026" into datetime
df2['Month_Year'] = pd.to_datetime(df2['Month_Year'], format='%B %Y')

# Aggregate contributions and validations
monthly_summary = df2.groupby('Month_Year')[['Contributions','Validations']].sum().reset_index()

# Sort by datetime
monthly_summary = monthly_summary.sort_values('Month_Year')

# Convert back to string for plotting labels
monthly_summary['Month_Year'] = monthly_summary['Month_Year'].dt.strftime('%b %Y')  # e.g., Jan 2026

print(monthly_summary)

# ✅ Grouped bar chart
fig = px.bar(
    monthly_summary,
    x='Month_Year',
    y=['Contributions','Validations'],
    barmode='group',
    text_auto=True,
    title="Monthly Contributions vs Validations"
)

fig.update_layout(
    xaxis_tickangle=-45,
    height=600
)

fig.show()

# ✅ Line chart comparison
fig_line = px.line(
    monthly_summary,
    x='Month_Year',
    y=['Contributions','Validations'],
    markers=True,
    title="Trend of Contributions vs Validations"
)

fig_line.update_layout(
    xaxis_tickangle=-45,
    height=600
)

fig_line.show()




  Month_Year  Contributions  Validations
0   Jan 2026           1085          865
1   Feb 2026           4831         4243
2   Mar 2026           3039         2751
3   Apr 2026           2160         2084


In [79]:
import pandas as pd

# Group by month and count unique languages
monthly_lang_count = df2.groupby('Month_Year')['Language'].nunique().reset_index()

monthly_lang_count.rename(columns={'Language':'Unique_Languages'}, inplace=True)

print(monthly_lang_count)


      Month_Year  Unique_Languages
0     April 2026                23
1  February 2026                21
2   January 2026                20
3     March 2026                21


In [17]:
# Ensure datetime and month-year column
df2['Activity_date'] = pd.to_datetime(df2['Activity_date'])
df2['Month_Year'] = df2['Activity_date'].dt.strftime('%B %Y')


In [26]:
import plotly.express as px

# Get all unique months
months = df2['Month_Year'].unique()

# Generate one plot per month
for month in months:
    # Filter data for this month
    month_data = df2[df2['Month_Year'] == month]
    
    # Group by date + language
    daily_lang_contrib = month_data.groupby(['Activity_date','Language'])['Contributions'].sum().reset_index()
    daily_lang_contrib = daily_lang_contrib[daily_lang_contrib['Contributions'] > 0]
    
    fig = px.bar(
        daily_lang_contrib,
        x='Activity_date',
        y='Contributions',
        color='Language',
        title=f"Daily Language Contributions - {month}",
        barmode='stack'
    )
    
    fig.update_layout(xaxis_title="Date", yaxis_title="Contributions", height=500)
    fig.show()   # use st.plotly_chart(fig) if inside Streamlit


In [19]:
# Ensure datetime and month-year column
df2['Activity_date'] = pd.to_datetime(df2['Activity_date'])
df2['Month_Year'] = df2['Activity_date'].dt.strftime('%B %Y')


In [22]:
# Group by month, date, and language
monthly_daily = df2.groupby(['Month_Year','Activity_date','Language'])['Contributions'].sum().reset_index()

# For each month, find the row with the maximum contributions
top_lang_per_month = monthly_daily.loc[monthly_daily.groupby('Month_Year')['Contributions'].idxmax()]
print(top_lang_per_month)


        Month_Year Activity_date Language  Contributions
60      April 2026    2026-04-10  English            112
139  February 2026    2026-02-02     Odia            371
335   January 2026    2026-01-20  English            108
551     March 2026    2026-03-27   Telugu            380


In [49]:
# Ensure datetime
df1['Activity_date'] = pd.to_datetime(df1['Activity_date'])
df1['Month_Year'] = df1['Activity_date'].dt.to_period('M')

# Sum contributions across all languages for each date
daily_totals = df1.groupby('Activity_date')[['New_users','Contributors']].sum().reset_index()

# Sort descending and take top 5
top5_dates = daily_totals.sort_values('Contributors', ascending=False).head(5)

# --- Add conversion rate (% of registered users who contributed) ---
top5_dates['User_to_Contributor(%)'] = (top5_dates['Contributors'] / top5_dates['New_users']) * 100

# --- Add monthly share (% of total contributions in that month) ---
# Monthly totals
monthly_contrib_totals = df1.groupby('Month_Year')['Contributors'].sum().reset_index()
monthly_contrib_totals = monthly_contrib_totals.rename(columns={'Contributors':'Monthly_total_contrib'})

# Merge back
top5_dates = top5_dates.merge(
    df1[['Activity_date','Month_Year']], on='Activity_date', how='left'
).merge(monthly_contrib_totals, on='Month_Year', how='left')

top5_dates['Day_share_of_month'] = (top5_dates['Contributors'] / top5_dates['Monthly_total_contrib']) * 100

print(top5_dates[['Activity_date','New_users','Contributors',
                  'User_to_Contributor(%)','Day_share_of_month']])


  Activity_date  New_users  Contributors  User_to_Contributor(%)  \
0    2026-02-21         30            18               60.000000   
1    2026-02-16         30            17               56.666667   
2    2026-02-15         27            17               62.962963   
3    2026-03-18         25            17               68.000000   
4    2026-02-17         30            16               53.333333   

   Day_share_of_month  
0            5.572755  
1            5.263158  
2            5.263158  
3            6.007067  
4            4.953560  


In [27]:
import plotly.express as px

# Ensure datetime and Month-Year column
df2['Activity_date'] = pd.to_datetime(df2['Activity_date'])
df2['Month_Year'] = df2['Activity_date'].dt.strftime('%B %Y')

# Get all unique months
months = df2['Month_Year'].unique()

# Generate one treemap per month
for month in months:
    # Filter data for this month
    month_data = df2[df2['Month_Year'] == month]
    
    # Group by date + language
    daily_lang_contrib = month_data.groupby(['Activity_date','Language'])['Contributions'].sum().reset_index()
    
    # Remove zero-contribution rows
    daily_lang_contrib = daily_lang_contrib[daily_lang_contrib['Contributions'] > 0]
    
    # ✅ Treemap instead of stacked bar
    fig = px.treemap(
        daily_lang_contrib,
        path=['Activity_date','Language'],   # hierarchy: date → language
        values='Contributions',
        title=f"Treemap of Daily Language Contributions - {month}"
    )
    
    fig.update_layout(height=600)
    fig.show()   # use st.plotly_chart(fig) if inside Streamlit


In [29]:
# Day with most new users
max_reg_day = df1.loc[df1['New_users'].idxmax()]

# Day with least new users
min_reg_day = df1.loc[df1['New_users'].idxmin()]
print("Most registrations:", max_reg_day['Activity_date'], max_reg_day['New_users'])
print("Least registrations:", min_reg_day['Activity_date'], min_reg_day['New_users'])

Most registrations: 2026-02-16 00:00:00 30
Least registrations: 2026-01-26 00:00:00 6


In [30]:
# Day with most contributors
max_contrib_day = df1.loc[df1['Contributors'].idxmax()]

# Day with least contributors
min_contrib_day = df1.loc[df1['Contributors'].idxmin()]
print("Most contributors:", max_contrib_day['Activity_date'], max_contrib_day['Contributors'])
print("Least contributors:", min_contrib_day['Activity_date'], min_contrib_day['Contributors'])

Most contributors: 2026-02-21 00:00:00 18
Least contributors: 2026-01-26 00:00:00 2


In [36]:
# Ensure datetime
df1['Activity_date'] = pd.to_datetime(df1['Activity_date'])
df1['Month_Year'] = df1['Activity_date'].dt.strftime('%B %Y')

# Aggregate by month
monthly_summary = df1.groupby('Month_Year')[['New_users','Contributors']].sum().reset_index()
print("\nMonthly consolidated summary:")
print(monthly_summary)
fig = px.bar(
    monthly_summary,
    x="Month_Year",
    y="Contributors",
    title="Contributors per Month",
    text="Contributors"   # show values on bars
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Number of Contributors",
    height=500
)

fig.show()


Monthly consolidated summary:
      Month_Year  New_users  Contributors
0     April 2026        306           166
1  February 2026        589           323
2   January 2026        230           117
3     March 2026        475           283


In [37]:
# Total contributions per language
lang_contrib_totals = df2.groupby('Language')['Contributions'].sum().reset_index()

# Total validations per language
lang_valid_totals = df2.groupby('Language')['Validations'].sum().reset_index()

# Language with max contributions
top_contrib = lang_contrib_totals.loc[lang_contrib_totals['Contributions'].idxmax()]

# Language with max validations
top_valid = lang_valid_totals.loc[lang_valid_totals['Validations'].idxmax()]

print("\nLanguage with most contributions:", top_contrib['Language'], top_contrib['Contributions'])
print("Language with most validations:", top_valid['Language'], top_valid['Validations'])

# Merge contributions and validations totals
lang_summary = pd.merge(lang_contrib_totals, lang_valid_totals, on='Language', how='outer').fillna(0)

# ✅ Add percentage column (share of total contributions + validations)
lang_summary['Total'] = lang_summary['Contributions'] + lang_summary['Validations']
lang_summary['Pct_of_Total'] = (lang_summary['Total'] / lang_summary['Total'].sum()) * 100

# Clean index
df_clean = lang_summary.reset_index(drop=True)

# Sort by contributions or validations
df_clean = df_clean.sort_values(['Contributions','Validations'], ascending=False)

print(df_clean)
import plotly.express as px

# Assuming your DataFrame is called df_clean
# and it has columns: Language, Contributions, Validations, Total, Pct_of_Total

fig = px.pie(
    df_clean,
    names="Language",
    values="Total",   # use combined contributions + validations
    title="Language Share of Total Activity",
    hole=0.3          # makes it a donut chart for readability
)

# Add percentage labels
fig.update_traces(
    textinfo="label+percent",   # show both language and percentage
    pull=[0.05]*len(df_clean),  # slightly pull out all slices for clarity
)

# Improve layout
fig.update_layout(
    height=600,
    legend_title="Languages"
)

fig.show()




Language with most contributions: English 3524
Language with most validations: English 2930
     Language  Contributions  Validations  Total  Pct_of_Total
5     English           3524         2930   6454     30.648685
7       Hindi           1782          903   2685     12.750499
23     Telugu           1108          747   1855      8.809004
9    Kashmiri           1055         1555   2610     12.394339
0    Assamese            612           25    637      3.024979
6    Gujarati            563         1439   2002      9.507076
17       Odia            481          278    759      3.604331
14    Marathi            367          232    599      2.844525
20    Santali            340          260    600      2.849273
12  Malayalam            307           77    384      1.823535
22      Tamil            261          123    384      1.823535
1     Bengali            150          125    275      1.305917
8     Kannada            120          118    238      1.130212
18    Punjabi            

In [84]:
lang_summary = df2.groupby('Language')[['Contributions','Validations']].sum().reset_index()

# Remaining = Contributions - Validations
lang_summary['Remaining'] = lang_summary['Contributions'] - lang_summary['Validations']

# Percentage of contributions still pending validation
lang_summary['Pct_Remaining'] = (lang_summary['Remaining'] / lang_summary['Contributions']) * 100

# Display as table
print(lang_summary)

# Totals
total_contrib = lang_summary['Contributions'].sum()
total_valid = lang_summary['Validations'].sum()
overall_remaining = total_contrib - total_valid
overall_pct_remaining = (overall_remaining / total_contrib) * 100

print("\n=== Overall Totals ===")
print("Total Contributions:", total_contrib)
print("Total Validations:", total_valid)
print("Remaining (not validated):", overall_remaining)
print("Percentage Remaining:", round(overall_pct_remaining, 2), "%")



     Language  Contributions  Validations  Remaining  Pct_Remaining
0    Assamese            612           25        587      95.915033
1     Bengali            150          125         25      16.666667
2       Bhili              0            5         -5           -inf
3        Bodo             38          538       -500   -1315.789474
4       Dogri             67          298       -231    -344.776119
5     English           3524         2930        594      16.855846
6    Gujarati            563         1439       -876    -155.595027
7       Hindi           1782          903        879      49.326599
8     Kannada            120          118          2       1.666667
9    Kashmiri           1055         1555       -500     -47.393365
10    Konkani             20            8         12      60.000000
11   Maithili              0          128       -128           -inf
12  Malayalam            307           77        230      74.918567
13   Manipuri              4            1       